# pyAuger Tutorial

This notebook walks through the complete workflow for computing direct Auger recombination coefficients ($C_n$ and $C_p$) from VASP first-principles data.

---

## Table of Contents

| # | Section | Description |
|---|---------|---|
| 1 | [Background](#1.-Background) | What Auger recombination is and what this code computes |
| 2 | [Prerequisites](#2.-Prerequisites) | Required VASP files, Python dependencies, and INCAR settings |
| 3 | [Configuration](#3.-Configuration) | Set paths and calculation parameters |
| 4 | [Step 0 — Manual band assignment](#4.-Step-0-—-Manual-Band-Assignment-(Optional)) | For zero-gap or narrow-gap materials |
| 5 | [Step 1 — Parse VASP data](#5.-Step-1-—-Parse-VASP-Data) | Read VASP outputs into pyAuger format |
| 6 | [Step 2 — Import parsed data](#6.-Step-2-—-Import-Parsed-Data) | Load the parsed data before any calculation |
| 7 | [Step 3 — Carrier concentrations](#7.-Step-3-—-Carrier-Concentrations) | Compute Fermi levels, $n$, $p$, and occupations |
| 8 | [Step 4 — Energy windows](#8.-Step-4-—-Energy-Window-Selection-(Optional)) | Determine CB/VB energy cutoffs |
| 9 | [Step 5 — Pair generation](#9.-Step-5-—-Pair-Generation) | Identify Auger scattering channels |
| 10 | [Step 6 — Matrix elements](#10.-Step-6-—-Matrix-Elements) | Compute Coulomb $|M|^2$ from wavefunctions |
| 11 | [Step 7 — Auger coefficients](#11.-Step-7-—-Auger-Coefficients) | Evaluate $C_n$ and $C_p$ |
| 12 | [Cheat sheet](#12.-Cheat-Sheet) | Copy-paste minimal workflow |


---
## 1. Background

### What is Auger recombination?

Auger recombination is a **non-radiative** carrier recombination mechanism in semiconductors. When an electron–hole pair recombines, the released energy is transferred to a **third carrier** instead of being emitted as a photon.

There are two types:

| Type | Label | Process | Coefficient |
|------|-------|---------|-------------|
| **eeh** | electron–electron–hole | Two CB electrons scatter; one recombines with a VB hole. The other CB electron gains the energy. | $C_n$ |
| **ehh** | electron–hole–hole | A CB electron recombines with a VB hole. A second VB hole gains the energy. | $C_p$ |

The total Auger coefficient is $C_\text{Auger} = C_n + C_p$.

### What does this code compute?

Starting from VASP DFT outputs (eigenvalues, wavefunctions, k-grid), the code:

1. **Identifies** the most important 4-particle scattering channels (“Auger pairs”), ranked by Fermi–Dirac occupation probability.
2. **Calculates** the screened Coulomb matrix element $|M|^2$ for each pair using the actual wavefunctions.
3. **Evaluates** the Auger coefficient via Fermi’s Golden Rule:

$$C_n = \frac{4\pi}{\hbar} \frac{1}{n^2 p - n_i^2 n} \sum_\text{pairs} P \cdot |M|^2 \cdot \delta(\Delta E)$$

where $P$ is the occupation-weighted probability and $\delta(\Delta E)$ enforces energy conservation.

---
## 2. Prerequisites

### Required VASP output files

Run a standard SCF (or hybrid-functional) calculation and collect these files in one directory:

| File | Purpose |
|------|---------|
| `EIGENVAL` | Band eigenvalues at every k-point |
| `vasprun.xml` | Dielectric tensor, structure, Fermi level |
| `KPOINTS` | k-grid specification (Monkhorst–Pack) |
| `POSCAR` | Crystal structure |
| `WAVECAR` | Plane-wave coefficients (needed for matrix elements) |

### VASP INCAR settings

The choice of **approach** affects which VASP settings you need:

- **`nearest_kpoint` approach** — requires `ISYM = -1` in the INCAR so that VASP writes all k-points in the full Brillouin zone (no symmetry reduction).
- **`exact_kpoint` approach** — does **not** require `ISYM = -1`, since it generates NSCF calculations at the exact k-points needed.

Suggested INCAR for the SCF run (nearest_kpoint approach):

```
PREC    = Accurate
ENCUT   = 400
EDIFF   = 1E-6
ISMEAR  = 0
SIGMA   = 0.05
ISYM    = -1        # Full BZ (required for nearest_kpoint)
LWAVE   = .TRUE.    # Write WAVECAR
LREAL   = .FALSE.
```

### Python dependencies

```
numpy, scipy, pandas, matplotlib, pymatgen, pyvaspwfc
```

In [ ]:
from auger import AugerCalculator, AugerAnalyzer, utilities

print("pyAuger loaded successfully.")

---
## 3. Configuration

Edit the cell below to match your system. All subsequent cells use these variables.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
VASP_FOLDER   = "./vasp_scf"           # Directory with EIGENVAL, vasprun.xml, KPOINTS, POSCAR, WAVECAR
RESULTS_DIR   = "./results"            # Where parsed data and results will be saved

# ── Physical parameters ────────────────────────────────────────────────────
TEMPERATURE    = 300                   # Temperature (K)
DOPING         = 0                     # Doping concentration (cm⁻³); positive = n-type, negative = p-type, 0 = intrinsic
EXCESS_CARRIER = 1e17                  # Excess carrier concentration Δn (cm⁻³)

# ── Material parameters ────────────────────────────────────────────────────
DIELECTRIC     = 16.8                  # Static dielectric constant (εᵣ)

# ── Energy windows ─────────────────────────────────────────────────────────
CB_WINDOW      = 0.3                   # Energy window above CBM (eV)
VB_WINDOW      = 0.3                   # Energy window below VBM (eV)

---
## 4. Step 0 — Manual Band Assignment (Optional)

For semimetals or materials where pymatgen detects a zero or incorrect band gap, the automatic identification of the conduction band minimum (CBM) and valence band maximum (VBM) may fail.

In such cases, manually specify the band indices **before** parsing. The indices are 0-based (i.e., the first band is index 0).

In [ ]:
calc = AugerCalculator(T=TEMPERATURE, nd=DOPING)

# Uncomment the lines below only if automatic band detection fails:
#
# calc.assign_firstCB_and_lastVB(
#     firstCB_index=25,    # 0-based index of the first conduction band
#     lastVB_index=24,     # 0-based index of the last valence band
# )

---
## 5. Step 1 — Parse VASP Data

This step reads the VASP output files and extracts the band structure. It only needs to be run **once** per VASP calculation. It reads `EIGENVAL`, `vasprun.xml`, `KPOINTS`, and `POSCAR`, then saves parsed `.npy` arrays and `band_info.txt` to disk.

All energies are referenced so that VBM = 0 eV.

In [ ]:
calc.parse_BS_data(
    folder_path=VASP_FOLDER,
    write_path=RESULTS_DIR,
    scissor_shift=0.0,       # Rigid shift applied to all CB bands (eV)
    # force_gap=1.5,         # Alternatively, force the gap to an exact value (overrides scissor_shift)
)

After parsing, the results directory contains:

```
results/
├── band_info.txt          ← material metadata (name, gap, lattice, etc.)
├── kgrid_X_XX.npy         ← k-point Cartesian coordinates
├── Egrid_X_XX.npy         ← eigenvalues (eV), VBM = 0
└── kw_X_XX.npy            ← k-point weights
```

Here `X` is the k-grid dimension and `XX` is the total number of k-points.

---
## 6. Step 2 — Import Parsed Data

This step loads the previously parsed `.npy` files and `band_info.txt` into the calculator. It **must** be called before any calculation, whether the data was just parsed or parsed in a previous session.

In [ ]:
calc.import_parsed_BS_data(from_folder=RESULTS_DIR)

---
## 7. Step 3 — Carrier Concentrations

This step solves the charge-neutrality equation self-consistently to find the Fermi level, then computes:

- **n**, **p** — electron and hole concentrations
- **n_i** — intrinsic carrier concentration
- **E_Fn**, **E_Fp** — quasi-Fermi levels (when `delta_n > 0`)
- Fermi–Dirac occupation for every state

In [ ]:
fn, fp = calc.calculate_carrier_concentrations(
    delta_n=EXCESS_CARRIER,   # Excess carriers (cm⁻³); set to 0 for equilibrium
)

print(f"Electron concentration:  n  = {calc.n:.4e} cm⁻³")
print(f"Hole concentration:      p  = {calc.p:.4e} cm⁻³")
print(f"Intrinsic concentration: ni = {calc.ni:.4e} cm⁻³")
print(f"Quasi-Fermi (electrons): Efn = {calc.Efn:.4f} eV")
print(f"Quasi-Fermi (holes):     Efp = {calc.Efp:.4f} eV")

---
## 8. Step 4 — Energy-Window Selection (Optional)

Auger transitions involve states within a thermal energy window around the band edges. You can either set `CB_WINDOW` and `VB_WINDOW` manually (see [Configuration](#3.-Configuration)) or let the code determine them automatically.

In [ ]:
CB_auto, VB_auto = calc.calculate_energy_cutoffs(charge_threshold=0.99)

print(f"Auto-determined energy windows:")
print(f"  CB_window = {CB_auto:.4f} eV")
print(f"  VB_window = {VB_auto:.4f} eV")

# You can use CB_auto/VB_auto below, or stick with the manual CB_WINDOW/VB_WINDOW.

---
## 9. Step 5 — Pair Generation

This step identifies all valid 4-particle scattering channels and ranks them by occupation probability.

When states 1, 2, 3 are chosen, the 4th must satisfy **momentum conservation**: $\mathbf{k}_1 + \mathbf{k}_2 = \mathbf{k}_3 + \mathbf{k}_4$.

**Two approaches** are available for determining the 4th state:

| Approach | Description | Extra VASP runs? |
|----------|-------------|------------------|
| `nearest_kpoint` | Maps k₄ to the nearest k-point in the SCF grid | No |
| `exact_kpoint` | Runs NSCF calculations at the exact k₄ | Yes |

**Two search modes** control how pairs are enumerated:

| Mode | Description |
|------|-------------|
| `Max_Heap` | Priority queue — fast, extracts top-N pairs efficiently |
| `Brute_Force` | Exhaustive enumeration — useful for small grids or reference |

Run pair generation **once per Auger type** (`"eeh"` for $C_n$, `"ehh"` for $C_p$).

### 9a. Approach 1 — `nearest_kpoint` (recommended)

In [ ]:
gen_eeh = calc.create_auger_pairs(
    CB_window=CB_WINDOW,
    VB_window=VB_WINDOW,
    auger_type="eeh",
    approach="nearest_kpoint",
    search_mode="Max_Heap",
    num_top_pairs=1000,              # Keep top 1000 pairs, or "all"
)

### 9b. Approach 2 — `exact_kpoint`

This is a multi-stage workflow:

1. Generate the list of off-grid k-points that need NSCF calculations.
2. Create NSCF input directories for VASP.
3. Run the NSCF VASP jobs externally.
4. Create pairs using the NSCF results.

#### Stage 1 — Generate the required k-points

In [ ]:
# calc.create_exact_kpoint_list(
#     CB_window=CB_WINDOW,
#     VB_window=VB_WINDOW,
#     auger_type="eeh",
#     poscar_path=VASP_FOLDER + "/POSCAR",
#     search_mode="Brute_Force",
#     num_kpoints="all",
# )

#### Stage 2 — Create NSCF input directories

The k-points listed under the `k2_target_frac_mapped` column in the CSV from Stage 1 are the ones that need NSCF calculations.

In [ ]:
# utilities.create_nscf_inputs(
#     scf_folder=VASP_FOLDER,
#     nscf_folder=RESULTS_DIR + "/NSCF_eeh_inputs",
#     exact_kpoints_table=RESULTS_DIR + "/exact_kpoints_eeh_XX.csv",
#     auger_type="eeh",
#     num_kpoints_per_file="all",
# )
#
# # After this, run the NSCF VASP jobs, then proceed to Stage 3.

#### Stage 3 — Create pairs using NSCF results

In [ ]:
# NSCF_FOLDERS = [RESULTS_DIR + "/NSCF_eeh_inputs"]
#
# gen_eeh_exact = calc.create_auger_pairs(
#     CB_window=CB_WINDOW,
#     VB_window=VB_WINDOW,
#     auger_type="eeh",
#     approach="exact_kpoint",
#     search_mode="Brute_Force",
#     nscf_folders=NSCF_FOLDERS,
#     num_top_pairs="all",
#     exact_kpoints_csv=RESULTS_DIR + "/exact_kpoints_eeh_XX.csv",
# )

---
## 10. Step 6 — Matrix Elements

This step computes the screened Coulomb matrix element $|M|^2$ for each pair using the wavefunctions from the WAVECAR.

The matrix element includes direct, exchange, and interference terms:

$$|M|^2 = |M_d|^2 + |M_x|^2 + |M_d - M_x|^2$$

This step runs in **parallel** using Python multiprocessing and is typically the most computationally intensive part of the workflow.

In [ ]:
me = calc.calculate_matrix_elements(
    auger_type="eeh",
    wavecar_files=VASP_FOLDER + "/WAVECAR",
    dielectric_constant=DIELECTRIC,
    num_matrix_elements="all",          # Compute all pairs, or an integer to limit
)

### Resuming an interrupted calculation

If the job was interrupted, you can resume from the partial results file:

In [ ]:
# me = calc.calculate_matrix_elements(
#     auger_type="eeh",
#     wavecar_files=VASP_FOLDER + "/WAVECAR",
#     dielectric_constant=DIELECTRIC,
#     continue_from_files=RESULTS_DIR + "/eeh_matrix_elements_partial.jsonl",
# )

---
## 11. Step 7 — Auger Coefficients

The final step sums all pair contributions to obtain the Auger coefficient. The code evaluates this for multiple delta functions and FWHM values simultaneously, allowing you to assess the sensitivity to broadening.

Available delta functions: `Gaussian`, `Lorentzian`, `Rectangular`.

### Loading pairs and matrix elements from files

If you are continuing from a previous session, load the pairs and matrix elements from disk before computing rates:

In [ ]:
# calc.read_auger_pairs(RESULTS_DIR + "/auger_eeh_pairs_XX.csv")
# calc.read_matrix_elements(RESULTS_DIR + "/eeh_matrix_elements_XX.jsonl")

In [ ]:
results_eeh = calc.calculate_auger_rates(
    auger_type="eeh",
    delta_function=("Gaussian", "Lorentzian"),
    FWHM=(0.01, 0.03, 0.05, 0.07, 0.1),
)

In [ ]:
import pandas as pd

df_eeh = pd.DataFrame(results_eeh)
print("EEH Auger Coefficients (C_n)")
print("=" * 55)
for delta in df_eeh["Delta function"].unique():
    print(f"\n  {delta}:")
    sub = df_eeh[df_eeh["Delta function"] == delta]
    for _, row in sub.iterrows():
        print(f"    FWHM = {row['FWHM']:.3f} eV  ->  C_n = {row['Auger coefficient']:.4e} cm^6/s")

### Repeat for EHH ($C_p$)

The workflow is identical — change `auger_type` to `"ehh"` and repeat Steps 5–7.

In [ ]:
# gen_ehh = calc.create_auger_pairs(
#     CB_window=CB_WINDOW, VB_window=VB_WINDOW,
#     auger_type="ehh",
#     approach="nearest_kpoint",
#     search_mode="Max_Heap",
#     num_top_pairs=1000,
# )
#
# me_ehh = calc.calculate_matrix_elements(
#     auger_type="ehh",
#     wavecar_files=VASP_FOLDER + "/WAVECAR",
#     dielectric_constant=DIELECTRIC,
# )
#
# results_ehh = calc.calculate_auger_rates(
#     auger_type="ehh",
#     delta_function=("Gaussian", "Lorentzian"),
#     FWHM=(0.01, 0.03, 0.05, 0.07, 0.1),
# )

### Total Auger coefficient

The total coefficient is $C_\text{Auger} = C_n + C_p$:

In [ ]:
# df_ehh = pd.DataFrame(results_ehh)
# merged = pd.merge(df_eeh, df_ehh, on=["FWHM", "Delta function"], suffixes=("_eeh", "_ehh"))
# merged["C_total"] = merged["Auger coefficient_eeh"] + merged["Auger coefficient_ehh"]
#
# print("Total Auger Coefficient (C_n + C_p)")
# print("=" * 65)
# for delta in merged["Delta function"].unique():
#     print(f"\n  {delta}:")
#     sub = merged[merged["Delta function"] == delta]
#     for _, row in sub.iterrows():
#         print(f"    FWHM = {row['FWHM']:.3f} eV  ->  C_total = {row['C_total']:.4e} cm^6/s")

---
## 12. Cheat Sheet

### Minimal workflow (copy-paste ready)

```python
from auger import AugerCalculator

calc = AugerCalculator(T=300, nd=0)
calc.parse_BS_data(folder_path="./vasp_scf", write_path="./results")
calc.import_parsed_BS_data(from_folder="./results")
calc.calculate_carrier_concentrations(delta_n=1e17)

# C_n (eeh)
calc.create_auger_pairs(CB_window=0.3, VB_window=0.3, auger_type="eeh")
calc.calculate_matrix_elements(auger_type="eeh", wavecar_files="./vasp_scf/WAVECAR", dielectric_constant=16.8)
Cn = calc.calculate_auger_rates(auger_type="eeh")

# C_p (ehh)
calc.create_auger_pairs(CB_window=0.3, VB_window=0.3, auger_type="ehh")
calc.calculate_matrix_elements(auger_type="ehh", wavecar_files="./vasp_scf/WAVECAR", dielectric_constant=16.8)
Cp = calc.calculate_auger_rates(auger_type="ehh")
```

### Key parameters

| Parameter | Typical range | Effect |
|-----------|---------------|--------|
| `CB_window` / `VB_window` | 0.2–0.5 eV | Larger = more states, slower but more accurate |
| `num_top_pairs` | 500–5000 | More pairs = more accurate, slower |
| `search_mode` | `Max_Heap` / `Brute_Force` | `Max_Heap` is faster for large grids |
| `FWHM` | 0.03–0.1 eV | Physical range; coefficient should plateau here |
| k-grid density | 10³–20³ | Increase until the coefficient converges |

### Output files

| File | Content |
|------|---------|
| `band_info.txt` | Material metadata |
| `Egrid_X_XX.npy` / `kgrid_X_XX.npy` / `kw_X_XX.npy` | Eigenvalues, k-points, weights |
| `auger_{type}_pairs_XX.csv` | Generated Auger pairs |
| `{type}_matrix_elements_XX.jsonl` | Matrix elements |
| `Auger_coefficients_{type}_XX.csv` | Final Auger coefficients |